# Notebook 1: Building a Perceptron from Scratch

In this notebook we build a **perceptron** — the simplest artificial neural network — using **pure Python** (no external libraries). By the end you will understand:

1. What a perceptron is and where it came from
2. How it makes predictions (forward pass)
3. How it learns from mistakes (weight update rule)
4. How to train and test one on a toy classification task

---

## 1. What is a Perceptron?

The perceptron was invented by **Frank Rosenblatt** in 1958. It is the simplest possible neural network: a single artificial neuron that takes several inputs, multiplies each by a learned **weight**, adds a **bias**, and passes the result through a **step function** to produce a binary output.

```
         x1 --(w1)--+
                     |
         x2 --(w2)--+---> Sum + bias ---> step() ---> prediction (0 or 1)
                     |
         x3 --(w3)--+
```

**Key idea:** The perceptron can learn any **linearly separable** pattern through supervised learning.

## 2. Our Classification Task

We will teach the perceptron to distinguish **Vehicles** from **Living Beings** using three binary features:

| Feature | Meaning | Values |
|---------|---------|--------|
| Is Round | Does it have round parts? | 0 = No, 1 = Yes |
| Has Wheels | Does it have wheels? | 0 = No, 1 = Yes |
| Can Breathe | Can it breathe? | 0 = No, 1 = Yes |

**Labels:** Vehicle = 1, Living Being = 0

## 3. Step 1 - Initialise Weights and Bias

We start with **random** weights and bias. The perceptron will adjust them during training.

In [2]:
import random

# Set a seed so results are reproducible
random.seed(42)

# One weight per feature (3 features)
weights = [random.uniform(-1, 1), random.uniform(-1, 1), random.uniform(-1, 1)]
bias = random.uniform(-1, 1)
learning_rate = 0.1

print("Starting weights:", [round(w, 3) for w in weights])
print("Starting bias:   ", round(bias, 3))

Starting weights: [0.279, -0.95, -0.45]
Starting bias:    -0.554


## 4. Step 2 - Prepare Training Data

Each sample is a tuple of `(features_list, label)`.

In [3]:
training_data = [
    # Vehicles (label = 1)
    ([1, 1, 0], 1),  # Car: round wheels, has wheels, can't breathe
    ([0, 1, 0], 1),  # Truck: not round, has wheels, can't breathe
    ([1, 1, 0], 1),  # Bicycle: round wheels, has wheels, can't breathe
    ([0, 1, 0], 1),  # Bus: not round, has wheels, can't breathe

    # Living Beings (label = 0)
    ([1, 0, 1], 0),  # Human: round head, no wheels, can breathe
    ([0, 0, 1], 0),  # Dog: not round, no wheels, can breathe
    ([0, 0, 1], 0),  # Cat: not round, no wheels, can breathe
    ([1, 0, 1], 0),  # Bird: round body, no wheels, can breathe
]

print(f"Training set size: {len(training_data)} examples")
print(f"  Vehicles:       {sum(1 for _, l in training_data if l == 1)}")
print(f"  Living Beings:  {sum(1 for _, l in training_data if l == 0)}")

Training set size: 8 examples
  Vehicles:       4
  Living Beings:  4


## 5. Step 3 - The Prediction Function

Before we train, let's define the two pieces of a forward pass:

1. **Weighted sum:** `total = w1*x1 + w2*x2 + w3*x3 + bias`
2. **Step activation:** if `total >= 0` output 1, otherwise 0

In [4]:
def predict(features, weights, bias):
    """
    Compute the perceptron output for a single sample.

    Parameters
    ----------
    features : list of numbers  - input values [x1, x2, x3]
    weights  : list of numbers  - learned weights [w1, w2, w3]
    bias     : float            - learned bias term

    Returns
    -------
    int : 0 or 1
    """
    # Weighted sum (manual dot product)
    total = 0
    for x, w in zip(features, weights):
        total += x * w
    total += bias

    # Step activation
    return 1 if total >= 0 else 0


# Quick test with untrained weights
sample_features = [1, 1, 0]  # a car
print(f"Prediction for {sample_features}: {predict(sample_features, weights, bias)}")
print("(This is probably wrong - the perceptron hasn't learned yet!)")

Prediction for [1, 1, 0]: 0
(This is probably wrong - the perceptron hasn't learned yet!)


## 6. Step 4 - Training Loop

The **Perceptron Learning Algorithm** works as follows:

```
for each epoch:
    for each (features, target) in training_data:
        prediction = predict(features)
        error = target - prediction        # will be -1, 0, or +1
        if error != 0:
            for i in range(num_features):
                weights[i] += learning_rate * error * features[i]
            bias += learning_rate * error
```

We repeat until the perceptron makes **zero mistakes** on the training set (or we reach a maximum number of epochs).

In [5]:
max_epochs = 100
history = []  # track errors per epoch for later analysis

print("Training the Perceptron...")
print("-" * 60)

for epoch in range(max_epochs):
    errors = 0

    for features, target in training_data:
        # Forward pass
        total = (weights[0] * features[0]
                 + weights[1] * features[1]
                 + weights[2] * features[2]
                 + bias)
        prediction = 1 if total >= 0 else 0

        # Error
        error = target - prediction

        if error != 0:
            errors += 1
            # Update each weight
            weights[0] += learning_rate * error * features[0]
            weights[1] += learning_rate * error * features[1]
            weights[2] += learning_rate * error * features[2]
            bias += learning_rate * error

    history.append(errors)
    print(f"Epoch {epoch + 1:3d} | Errors: {errors} | "
          f"Weights: [{weights[0]:.3f}, {weights[1]:.3f}, {weights[2]:.3f}] | "
          f"Bias: {bias:.3f}")

    if errors == 0:
        print(f"\n>>> Perfect! Training converged in {epoch + 1} epoch(s).")
        break
else:
    print(f"\nReached {max_epochs} epochs without full convergence.")

Training the Perceptron...
------------------------------------------------------------
Epoch   1 | Errors: 4 | Weights: [0.479, -0.550, -0.450] | Bias: -0.154
Epoch   2 | Errors: 4 | Weights: [0.479, -0.250, -0.550] | Bias: 0.046
Epoch   3 | Errors: 3 | Weights: [0.379, -0.050, -0.650] | Bias: 0.146
Epoch   4 | Errors: 0 | Weights: [0.379, -0.050, -0.650] | Bias: 0.146

>>> Perfect! Training converged in 4 epoch(s).


### Training Error Over Epochs

Let's visualise how quickly the perceptron learned.

In [6]:
# Simple text-based chart
print("Errors per epoch:")
for i, e in enumerate(history):
    bar = "#" * e
    print(f"  Epoch {i+1:3d}: {bar} ({e})")

Errors per epoch:
  Epoch   1: #### (4)
  Epoch   2: #### (4)
  Epoch   3: ### (3)
  Epoch   4:  (0)


## 7. Step 5 - Test the Trained Perceptron

Let's give the perceptron some "mystery objects" it has never seen and check if it classifies them correctly.

In [7]:
test_cases = [
    ([1, 1, 0], "Mystery 1 - round, wheels, no breathing"),
    ([0, 0, 1], "Mystery 2 - not round, no wheels, breathes"),
    ([1, 0, 1], "Mystery 3 - round, no wheels, breathes"),
    ([0, 1, 0], "Mystery 4 - not round, wheels, no breathing"),
]

print("=" * 55)
print("Testing the Perceptron on unseen objects")
print("=" * 55)

for features, description in test_cases:
    pred = predict(features, weights, bias)
    label = "VEHICLE" if pred == 1 else "LIVING BEING"
    print(f"\n{description}")
    print(f"  Features : Is Round={features[0]}, Has Wheels={features[1]}, Can Breathe={features[2]}")
    print(f"  Prediction: {label}")

Testing the Perceptron on unseen objects

Mystery 1 - round, wheels, no breathing
  Features : Is Round=1, Has Wheels=1, Can Breathe=0
  Prediction: VEHICLE

Mystery 2 - not round, no wheels, breathes
  Features : Is Round=0, Has Wheels=0, Can Breathe=1
  Prediction: LIVING BEING

Mystery 3 - round, no wheels, breathes
  Features : Is Round=1, Has Wheels=0, Can Breathe=1
  Prediction: LIVING BEING

Mystery 4 - not round, wheels, no breathing
  Features : Is Round=0, Has Wheels=1, Can Breathe=0
  Prediction: VEHICLE


## 8. Inspect the Learned Weights

The final weights tell us which features the perceptron considers most important.

In [8]:
feature_names = ["Is Round", "Has Wheels", "Can Breathe"]

print("Final learned parameters:")
for name, w in zip(feature_names, weights):
    direction = "pushes toward Vehicle" if w > 0 else "pushes toward Living Being"
    print(f"  {name:12s}: {w:+.3f}  ({direction})")
print(f"  {'Bias':12s}: {bias:+.3f}")

print("\nInterpretation:")
print("  - A large positive weight for 'Has Wheels' makes sense - vehicles have wheels.")
print("  - A large negative weight for 'Can Breathe' makes sense - living beings breathe.")
print("  - 'Is Round' has a small weight because both categories have round examples.")

Final learned parameters:
  Is Round    : +0.379  (pushes toward Vehicle)
  Has Wheels  : -0.050  (pushes toward Living Being)
  Can Breathe : -0.650  (pushes toward Living Being)
  Bias        : +0.146

Interpretation:
  - A large positive weight for 'Has Wheels' makes sense - vehicles have wheels.
  - A large negative weight for 'Can Breathe' makes sense - living beings breathe.
  - 'Is Round' has a small weight because both categories have round examples.


## 9. Exercises for Students

1. **Add more training data** - create new objects (e.g., skateboard `[0,1,0]`, fish `[0,0,1]`). Does the perceptron still converge?
2. **Change the learning rate** - try `0.01` and `1.0`. How does it affect convergence speed?
3. **Try a non-linearly-separable problem** - can the perceptron learn XOR (`[0,0]->0, [0,1]->1, [1,0]->1, [1,1]->0`)? Why or why not?
4. **Track weight changes** - modify the training loop to store weights at each step and plot them.

---

## 10. Summary

| Concept | What we learned |
|---------|----------------|
| Perceptron | Simplest neural network - one neuron |
| Forward pass | Weighted sum + step activation |
| Learning rule | `w = w + lr * error * x` |
| Convergence | Guaranteed if data is linearly separable |
| Limitation | Cannot solve non-linearly-separable problems (e.g., XOR) |

**Next:** Open *Notebook 2* to see how NumPy makes this more efficient with vectorised operations.